# 04 · Thinking off (stage S2)

The chat template ignores `add_generation_prompt` and `reasoning_effort="none"` does not stop thinking, so the start of the
model reply is appended to the prompt (`prefill`). Variants:

- **A** `text`: opens the answer message.
- **B** `empty_think`: an empty thinking message, then the answer message.

Checks:
1. **Template:** both prefills tokenize to their special tokens.
2. **Inputs:** prefill only appends ids; raw output of baseline, A, B.
3. **Done-when:** the better variant parses at least 95% on 20 GQA and 20 TextVQA dev items.
4. **res_px:** 768 vs 1024 on TextVQA dev. You decide.

In [ ]:
N_ITEMS = 20
N_RES = 50
R = 3
MAX_NEW = 16
RES = [768, 1024]
PREFILLS = {
    "A": "<|message_model|><|content_text|>",
    "B": "<|message_model|><|content_thinking|><|end_message|><|message_model|><|content_text|>",
}
CONFIG = "configs/mvp.yaml"

In [ ]:
from adaloop import api
ROOT, cfg = api.setup_notebook(CONFIG)

CHECKS = []


def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok)))
    print(("PASS " if ok else "FAIL ") + f"{name}" + (f": {detail}" if detail else ""))
    assert ok, f"{name} failed: {detail}"


print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
vlm = api.load_model(cfg)
tok = vlm.processor.tokenizer

## 1 · Chat template and special tokens

In [ ]:
print("[STAGE] template")
print(vlm.processor.chat_template or tok.chat_template)

In [ ]:
for token, token_id in sorted(tok.get_added_vocab().items(), key=lambda kv: kv[1]):
    if "unused" not in token:
        print(token_id, token)
print("eos", tok.eos_token_id, tok.eos_token)

In [ ]:
question = [{"role": "user", "content": [{"type": "text", "text": "What color is the sky?"}]}]
print(repr(vlm.processor.apply_chat_template(question, reasoning_effort="none", add_generation_prompt=True, tokenize=False)))

In [ ]:
import re

for name, prefill in PREFILLS.items():
    ids = tok(prefill, add_special_tokens=False)["input_ids"]
    tokens = tok.convert_ids_to_tokens(ids)
    print(name, ids, tokens)
    check(f"S2 prefill {name} decodes back", tok.decode(ids, skip_special_tokens=False) == prefill)
    check(f"S2 prefill {name} one id per marker", tokens == re.findall(r"<\|[^|]+\|>", prefill), tokens)
print("✓ template")

## 2 · One item: baseline vs prefill

In [ ]:
print("[STAGE] one item")
item = api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0]
prompt = item.question + cfg["prompt_suffix"]
image = api.resize_long_side(item.image, cfg["res_px"])
print(repr(prompt), "->", item.answers)

base = api.prepare_inputs(vlm, prompt, image)
n = base["input_ids"].shape[-1]
inputs = {"baseline": base}
for name, prefill in PREFILLS.items():
    inputs[name] = api.prepare_inputs(vlm, prompt, image, prefill=prefill)
    ids = inputs[name]["input_ids"][0].tolist()
    check(f"S2 {name} first {n} ids unchanged", ids[:n] == base["input_ids"][0].tolist())
    check(f"S2 {name} tail is the prefill", ids[n:] == tok(prefill, add_special_tokens=False)["input_ids"], ids[n:])
    check(f"S2 {name} attention mask matches", inputs[name]["attention_mask"].shape == inputs[name]["input_ids"].shape
          and bool(inputs[name]["attention_mask"].all()))

In [ ]:
api.set_loops(vlm, R)
for name, inp in inputs.items():
    # baseline keeps the long limit so its thinking is visible
    max_new = cfg["max_new_tokens"] if name == "baseline" else MAX_NEW
    gen = api.generate(vlm, inp, max_new)
    print(f"--- {name}: {len(gen.token_ids)} tokens, hit_limit={gen.hit_limit}")
    print(repr(gen.raw_text))
    print("extracted:", repr(api.extract_answer(gen.raw_text, gen.hit_limit)))

In [ ]:
# with a prefill the generated text starts at the answer itself (no <|content_text|>)
cases = [  # (raw output, hit_limit, expected answer)
    ("blue<|end_message|><|content_model_end_sampling|>", False, "blue"),
    ("blue<|end_message|>", True, "blue"),
    ("a long answer that got cut", True, None),
]
for raw, hit, want in cases:
    got = api.extract_answer(raw, hit)
    check(f"S2 extract prefilled {raw[:20]!r} hit_limit={hit}", got == want, f"got {got!r}")
print("✓ one item")

## 3 · Parse rate of A and B (done-when)

`accuracy` and `parse_rate` are percentages.

In [ ]:
print("[STAGE] variants")
items = {
    "gqa": api.load_gqa(cfg, "dev", limit=N_ITEMS, root=ROOT),
    "textvqa": api.load_textvqa(cfg, "dev", limit=N_ITEMS, root=ROOT),
}

In [ ]:
import pandas as pd


def run(bench, bench_items, variant, res_px):
    """One evaluation with an in-memory config copy; the file name carries the config hash."""
    run_cfg = {**cfg, "answer_prefill": PREFILLS[variant], "max_new_tokens": MAX_NEW, "res_px": res_px}
    name = f"{bench}_dev_n{len(bench_items)}_res{res_px}_R{R}_{variant}_{api.config_hash(run_cfg)}.jsonl"
    out = ROOT / cfg["out_dir"] / "nb04" / name
    summary = api.run_evaluation(vlm, bench_items, run_cfg, R, out, resume=True)
    return summary, api.read_results(out)

In [ ]:
table, variant_rows = [], {}
for variant in PREFILLS:
    for bench, bench_items in items.items():
        summary, rows = run(bench, bench_items, variant, cfg["res_px"])
        variant_rows[variant, bench] = rows
        table.append({"variant": variant, "benchmark": bench, "n": summary["n"], "accuracy": summary["accuracy"],
                      "parse_rate": summary["parse_rate"], "n_out_tokens_mean": summary["n_out_tokens_mean"]})
table = pd.DataFrame(table)
table

In [ ]:
for (variant, bench), rows in variant_rows.items():
    print(f"--- {variant} {bench}")
    for r in rows[:3]:
        print(f"  {r['raw_output']!r} -> {r['pred']!r}  answers {r['answers'][:3]}")
    for r in [r for r in rows if not r["parse_ok"]][:3]:
        print(f"  UNPARSED {r['raw_output']!r}")

In [ ]:
worst = table.groupby("variant").parse_rate.min()  # worst benchmark per variant
best = "A" if worst["A"] >= worst["B"] else "B"
PREFILL = best  # edit to "A" or "B" to override; section 4 uses it
print("lowest parse rate per variant", worst.to_dict(), "| best", best, "| PREFILL", PREFILL)

In [ ]:
check("S2 all variant rows present", all(len(rows) == N_ITEMS for rows in variant_rows.values()))
check("S2 better variant parses >= 95% on both benchmarks", worst[best] >= 95.0, f"{best}: {worst[best]:.1f}")
print("✓ variants")

## 4 · res_px: 768 vs 1024 on TextVQA dev

`resize_long_side` never upscales: an image whose long side is already below `res_px` is sent as it is.
The processor does not resize; it makes one visual token per 40x40 patch, so `n_vis` should match the expected count.

In [ ]:
import math

print("[STAGE] res_px")
res_items = api.load_textvqa(cfg, "dev", limit=N_RES, root=ROOT)
long_sides = [max(it.image.size) for it in res_items]
print("original long side: min", min(long_sides), "max", max(long_sides))
patch = vlm.processor.image_processor.size
print("patch", patch)

expected = {}
for res_px in RES:
    sizes = [api.resize_long_side(it.image, res_px).size for it in res_items]
    expected[res_px] = sum(math.ceil(w / patch["width"]) * math.ceil(h / patch["height"]) for w, h in sizes) / len(sizes)
    print(f"res_px {res_px}: {sum(s < res_px for s in long_sides)}/{len(res_items)} images smaller than res_px, "
          f"largest sent {max(max(s) for s in sizes)}, expected mean n_vis {expected[res_px]:.1f}")

In [ ]:
res_table = []
for res_px in RES:
    summary, rows = run("textvqa", res_items, PREFILL, res_px)
    res_table.append({"res_px": res_px, "variant": PREFILL, "n": len(rows), "accuracy": summary["accuracy"],
                      "parse_rate": summary["parse_rate"], "n_vis_mean": summary["n_vis_mean"],
                      "n_vis_expected": expected[res_px], "generate_s_mean": summary["generate_s_mean"],
                      "peak_mem_gb_max": summary["peak_mem_gb_max"]})
res_table = pd.DataFrame(res_table)
res_table

In [ ]:
check("S2 res_px rows complete", (res_table.n == N_RES).all(), res_table.n.tolist())
print("✓ res_px")

## 5 · Decision to log

Paste into `docs/decisions.md`, fill in the blanks, then set `answer_prefill`, `res_px` and `max_new_tokens` in `configs/mvp.yaml`.

In [ ]:
parse = {bench: value for bench, value in table[table.variant == PREFILL].set_index("benchmark").parse_rate.items()}
print(f"## <date> · S2 · thinking off and res_px (git {api.git_hash()})")
print(f"- Decision: answer_prefill = variant {PREFILL} {PREFILLS[PREFILL]!r}, max_new_tokens = {MAX_NEW}; "
      f"parse rate GQA {parse['gqa']:.1f}, TextVQA {parse['textvqa']:.1f} (n={N_ITEMS} each, R={R}, res_px={cfg['res_px']}).")
summary_text = "; ".join(f"{r.res_px}: acc {r.accuracy:.1f}, n_vis {r.n_vis_mean:.0f}, {r.generate_s_mean:.2f}s, "
                         f"{r.peak_mem_gb_max:.1f} GB" for r in res_table.itertuples())
print(f"- Decision: res_px = <768|1024> because <reason>. TextVQA dev n={N_RES}: {summary_text}.")

## Summary

In [ ]:
pd.DataFrame(CHECKS, columns=["check", "pass"])